### Trabajo Integrador Final - Python para la Ciencia de Datos

**Prefesores:**

- Acosta, Gabriel
- Benitez, Flavian Dante

---

**Estudiantes:**

- Silvera, Matías
- Ayala, Lautaro
- Gaona, Axel

---

Análisis y predicción de precio de reventa de vehículos usados.


In [194]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

In [195]:
#Importamos el dataset y revisamos los primeros registros para entender su estructura y contenido
df = pd.read_csv('Car details v3.csv')
df.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.4 kmpl,1248 CC,74 bhp,190Nm@ 2000rpm,5.0
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14 kmpl,1498 CC,103.52 bhp,250Nm@ 1500-2500rpm,5.0
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.7 kmpl,1497 CC,78 bhp,"12.7@ 2,700(kgm@ rpm)",5.0
3,Hyundai i20 Sportz Diesel,2010,225000,127000,Diesel,Individual,Manual,First Owner,23.0 kmpl,1396 CC,90 bhp,22.4 kgm at 1750-2750rpm,5.0
4,Maruti Swift VXI BSIII,2007,130000,120000,Petrol,Individual,Manual,First Owner,16.1 kmpl,1298 CC,88.2 bhp,"11.5@ 4,500(kgm@ rpm)",5.0


In [196]:
print("\nDimensiones del dataset:")
print(df.shape)
print("\nTipos de datos:")
print(df.dtypes)


Dimensiones del dataset:
(8128, 13)

Tipos de datos:
name                 str
year               int64
selling_price      int64
km_driven          int64
fuel                 str
seller_type          str
transmission         str
owner                str
mileage              str
engine               str
max_power            str
torque               str
seats            float64
dtype: object


In [197]:
#Análisi de valores nulos en el dataset

datos_nulos = df.isnull().sum()
perdidos_pct = (datos_nulos / len(df) * 100).round(2)
perdidos_df = pd.DataFrame({
    'Cantidad de datos nulos': datos_nulos,
    '% Perdida': perdidos_pct
})
print(perdidos_df[perdidos_df['Cantidad de datos nulos'] > 0])

           Cantidad de datos nulos  % Perdida
mileage                        221       2.72
engine                         221       2.72
max_power                      215       2.65
torque                         222       2.73
seats                          221       2.72


**Hallazgo**: Se detectaron datos nulos en las columnas
- mileage
- engine
- max_power
- torque
- seats

**Decisión**: Se eliminarán los datos debido a que representan un porcentaje ínfimo del dataset y utilizar técnicas de imputación podría llevar el riesgo de introducir un sesgo en la varianza del modelo.

In [198]:
# Verificación de estado de las columnas

for col in ["mileage", "engine", "max_power", "torque"]:
    if col in df.columns:
        print(f" {col}: {df[col].dropna().head(3).tolist()} ")

import re
def extraer_numero(value):
    if pd.isna(value):
        return np.nan
    val = str(value).strip().lower()

    match = re.search(r'([0-9]*\.?[0-9]+)', val)
    if match:
        return float(match.group(1))
    else:
        return np.nan

 mileage: ['23.4 kmpl', '21.14 kmpl', '17.7 kmpl'] 
 engine: ['1248 CC', '1498 CC', '1497 CC'] 
 max_power: ['74 bhp', '103.52 bhp', '78 bhp'] 
 torque: ['190Nm@ 2000rpm', '250Nm@ 1500-2500rpm', '12.7@ 2,700(kgm@ rpm)'] 


In [199]:
# Extraemos los números de las columnas problemáticas
numeric_cols = ["mileage", "engine", "max_power"]
for col in numeric_cols:
    if col in df.columns:
        df[col] = df[col].apply(extraer_numero)

# Verificamos los resultados
for col in numeric_cols:
    if col in df.columns:
        print(f" {col}: {df[col].dropna().head(3).tolist()} ")

 mileage: [23.4, 21.14, 17.7] 
 engine: [1248.0, 1498.0, 1497.0] 
 max_power: [74.0, 103.52, 78.0] 


In [200]:
#Función para tratar valores atípicos
def remover_outliers_iqr(df, column, multiplier=3.0):
    Q1= df[column].quantile(0.25)
    Q3= df[column].quantile(0.75)
    IQR= Q3 - Q1
    limite_inferior= Q1 - multiplier * IQR
    limite_superior= Q3 + multiplier * IQR
    
    return df[(df[column] >= limite_inferior) & (df[column] <= limite_superior)]

In [201]:
# Aplicamos la función a las columnas relevantes
for col in ["selling_price", "km_driven", "engine", "max_power"]:
    if col in df.columns:
        df = remover_outliers_iqr(df, col)

*Anteriormente se usaba un multiplicador de 1.5 para establecer límites en cuanto al rango de valores permitidos dentro del dataset, pero recortaba aproximadamente el 28% de los datos (Lo cual es mucho), se optó por aumentar el multiplicador a 3.0 para eliminar solo aquellos casos en donde los valores sean extremos.*

**nota**: Se identificó que el dataset proviene de la India, en donde la mayoría de autos son económicos, por lo que con el primer recorte del 28% se eliminan todos aquellos vehículos de gama alta, lo que podría hacer que el modelo final prediga precios muy bajos para vehículos de gama alta o de lujo. 

In [202]:
df.isnull().sum()
# Gracias a que eliminamos los outliers ya no tenemos filas con valores nulos.

name             0
year             0
selling_price    0
km_driven        0
fuel             0
seller_type      0
transmission     0
owner            0
mileage          0
engine           0
max_power        0
torque           0
seats            0
dtype: int64

In [203]:
# Mejorando la feature engineering

df['car_age'] = 2024 - df['year'] #El dataset es del año 2024, por lo que restamos el año de fabricación para obtener la edad del vehículo

#Extraccion de la marca del vehículo a partir del nombre
def extraer_marca(nombre):
    name = str(nombre).strip()

    multi_words_brands = ['Land Rover', 'Alfa Romeo', 'Aston Martin']

    for brand in multi_words_brands:
        if name.lower().startswith(brand.lower()):
            return brand
    return name.split()[0]

df["brand"] = df["name"].apply(extraer_marca)

In [204]:
df.head(3)

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,mileage,engine,max_power,torque,seats,car_age,brand
0,Maruti Swift Dzire VDI,2014,450000,145500,Diesel,Individual,Manual,First Owner,23.40,1248.0,74.00,190Nm@ 2000rpm,5.0,10,Maruti
1,Skoda Rapid 1.5 TDI Ambition,2014,370000,120000,Diesel,Individual,Manual,Second Owner,21.14,1498.0,103.52,250Nm@ 1500-2500rpm,5.0,10,Skoda
2,Honda City 2017-2020 EXi,2006,158000,140000,Petrol,Individual,Manual,Third Owner,17.70,1497.0,78.00,"12.7@ 2,700(kgm@ rpm)",5.0,18,Honda


In [205]:
# Los datos de precios tienen una distribución sesgada a la derecha, 
# por lo que aplicamos una transformación logarítmica para normalizarla y mejorar el rendimiento de los modelos de regresión.
df['selling_price_log'] = np.log1p(df['selling_price'])

df['km_per_year'] = df['km_driven'] / (df['car_age'] + 1)

# 3. Potencia Específica(BHP por Litro)
df['specific_power'] = df['max_power'] / (df['engine'] / 1000)

df[['selling_price', 'selling_price_log', 'km_per_year', 'specific_power']].head(3)

,selling_price,selling_price_log,km_per_year,specific_power
0,450000,13.017005,13227.272727,59.294872
1,370000,12.821261,10909.090909,69.105474
2,158000,11.970357,7368.421053,52.104208


### Feature Engineering: Creación de Variables Clave

Decidimos crear tres nuevas métricas a partir de los datos existentes. El objetivo de este paso es sintetizar variables que representen mejor la realidad del mercado automotriz, facilitándole al modelo la detección de patrones de valor:

1. **Transformación Logarítmica del Precio (`selling_price_log`):** Los precios de los vehículos tienen una distribución naturalmente sesgada (hay muchísimos autos económicos y muy pocos de altísima gama). Al aplicar el logaritmo (`np.log1p`), "suavizamos" la curva hacia una distribución normal, lo que estabiliza la varianza y reduce el margen de error de nuestras predicciones.
2. **Intensidad de Uso (`km_per_year`):**Un auto con 100.000 km en 10 años tuvo un desgaste moderado, pero ese mismo kilometraje en solo 2 años indica un uso intensivo (posiblemente comercial o taxi). Esta variable le otorga al modelo una métrica mucho más precisa sobre el nivel de desgaste real del vehículo.
3. **Potencia Específica (`specific_power`):** Al convertir la cilindrada a litros y dividir la potencia máxima por este valor, obtenemos un índice de eficiencia del motor. Esto ayuda al algoritmo a diferenciar motores antiguos o ineficientes de mecánicas modernas (como motores turbo), las cuales el mercado suele cotizar a un precio mayor.

### One-Hot Encoding

In [210]:
#Definimos la 'y' (Variable Objetivo)
y = df['selling_price_log']

#Definimos la 'X' cruda (Eliminando lo que no sirve)
X_crudo = df.drop(columns=['name', 'torque', 'selling_price', 'selling_price_log'])

#Aplicamos One-Hot Encoding
columnas_categoricas = ['fuel', 'seller_type', 'transmission', 'owner', 'brand']
X = pd.get_dummies(X_crudo, columns=columnas_categoricas, drop_first=True)

X.columns

Index(['year', 'km_driven', 'mileage', 'engine', 'max_power', 'seats',
       'car_age', 'km_per_year', 'specific_power', 'fuel_Diesel', 'fuel_LPG',
       'fuel_Petrol', 'seller_type_Individual', 'seller_type_Trustmark Dealer',
       'transmission_Manual', 'owner_Fourth & Above Owner',
       'owner_Second Owner', 'owner_Test Drive Car', 'owner_Third Owner',
       'brand_Ashok', 'brand_Audi', 'brand_BMW', 'brand_Chevrolet',
       'brand_Daewoo', 'brand_Datsun', 'brand_Fiat', 'brand_Force',
       'brand_Ford', 'brand_Honda', 'brand_Hyundai', 'brand_Isuzu',
       'brand_Jeep', 'brand_Kia', 'brand_Land Rover', 'brand_MG',
       'brand_Mahindra', 'brand_Maruti', 'brand_Mercedes-Benz',
       'brand_Mitsubishi', 'brand_Nissan', 'brand_Opel', 'brand_Renault',
       'brand_Skoda', 'brand_Tata', 'brand_Toyota', 'brand_Volkswagen',
       'brand_Volvo'],
      dtype='str')

### Ajuste del Modelo Gradient Boosting

In [211]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error
import joblib

#División de los datos 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Instanciación y Entrenamiento del Modelo
modelo_gb = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=42)
modelo_gb.fit(X_train, y_train)

# Predicción sobre datos nunca vistos
y_pred = modelo_gb.predict(X_test)

#Medición del rendimiento
precision_r2 = r2_score(y_test, y_pred)
error_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Resultados:")
print(f"R^2 (Precisión de acierto): {precision_r2 * 100:.2f}%")
print(f"RMSE (Margen de error logarítmico promedio): {error_rmse:.4f}")



Resultados:
R^2 (Precisión de acierto): 92.49%
RMSE (Margen de error logarítmico promedio): 0.1907


In [ ]:
#Exportación del modelo y las columnas para su uso en el backend
joblib.dump(modelo_gb, 'modelo_gb_autos.pkl')
joblib.dump(list(X.columns), 'columnas_autos.pkl')

print("\n✅ Archivos .pkl generados con éxito.")